# 02 — Data Preparation
**Ferngill & Co. | LUBS5308M practice assessment | CRISP-DM phase 3**

In Step 1 we *found* the data-quality issues. In this notebook we *fix* them, one at a time, and save a clean file for modelling.

**Golden rules we follow**
1. Never edit the original file — we work on a copy.
2. Never silently change data — every change gets a **flag column** so we can see which rows were touched.
3. Prefer **flagging** over **deleting** — with only 250 customers, every row matters.

| Step | Issue (from Step 1) | Treatment |
|---|---|---|
| 2.1 | Missing `email_engagement` (6) | Fill with median of the customer's subscription group + flag |
| 2.2 | Missing `satisfaction_rating` (8, mostly subscribers) | Fill with median of the customer's subscription group + flag |
| 2.3 | `distinct_product_types` outlier (12) | Cap at 9 + flag |
| 2.4 | `average_order_value` exactly £5.00 | Keep + flag |
| 2.5 | Tenure < 12 months (part-year value) | Keep + flag |
| 2.6 | Tenure < 12 months but > 2 orders per month | Keep + flag |
| 2.7 | Create model-ready columns | `is_subscriber`, `high_value` |

**Output:** `data_clean/ferngill_clean.csv` and `outputs/tables/02_cleaning_log.csv`

## Setup

In [1]:
import pandas as pd
import os

# File locations (relative to this notebook, which lives in the 'notebooks' folder)
RAW_FILE   = "../dataset/ferngill_customers.csv"
CLEAN_FILE = "../data_clean/ferngill_clean.csv"
LOG_FILE   = "../outputs/tables/02_cleaning_log.csv"

# Make sure the output folders exist
os.makedirs("../data_clean", exist_ok=True)
os.makedirs("../outputs/tables", exist_ok=True)

## Load the raw data and make a working copy

In [2]:
raw = pd.read_csv(RAW_FILE)

# We work on a copy so the original stays untouched
df = raw.copy()

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
df.head()

Rows: 250
Columns: 13


,customer_id,age,region,acquisition_channel,tenure_months,total_orders,average_order_value,distinct_product_types,discount_sensitivity,email_engagement,satisfaction_rating,subscription_status,customer_value_12m
0,FG0001,32,Midlands,Marketplace,6,13,24.00,4,0.413,0.0,4.7,no,242.68
1,FG0002,33,Scotland & Wales,Instagram,48,26,56.96,8,0.164,70.2,4.8,yes,1833.39
2,FG0003,53,North,Google,3,5,31.35,3,0.051,33.1,4.5,no,169.39
3,FG0004,37,Scotland & Wales,Google,38,3,53.40,2,0.018,14.6,5.0,no,157.58
4,FG0005,36,Scotland & Wales,Google,16,7,19.94,4,0.214,50.6,5.0,no,114.05


In [3]:
# How many values are missing in each column?
df.isnull().sum()

customer_id               0
age                       0
region                    0
acquisition_channel       0
tenure_months             0
total_orders              0
average_order_value       0
distinct_product_types    0
discount_sensitivity      0
email_engagement          6
satisfaction_rating       8
subscription_status       0
customer_value_12m        0
dtype: int64

In [4]:
# A simple list where we record every change we make (this becomes our cleaning log)
cleaning_log = []

## 2.1 & 2.2 Missing values

**Why not just use the overall median?**
In Step 1 we found that **6 of the 8 missing satisfaction ratings belong to subscribers**, and subscribers are mostly our high-value customers.
Subscribers and non-subscribers have different typical values, so filling every gap with one overall number would pull
our best customers towards the average of everyone else.

So we fill each gap with the median **of the customer's own group** (subscriber or not).
We use the **median** rather than the mean because it is not pulled around by extreme values.

**Why not delete these rows?** We would lose 14 customers (5.6% of the data) — mostly high-value ones, which is exactly the group we are studying.

In [5]:
# First, look at the typical values in each group
print("Median email engagement by group:")
print(df.groupby("subscription_status")["email_engagement"].median())

print("\nMedian satisfaction rating by group:")
print(df.groupby("subscription_status")["satisfaction_rating"].median())

print("\nOverall medians (the 'simple' option we are NOT using):")
print("email_engagement:   ", df["email_engagement"].median())
print("satisfaction_rating:", df["satisfaction_rating"].median())

Median email engagement by group:
subscription_status
no     38.35
yes    75.40
Name: email_engagement, dtype: float64

Median satisfaction rating by group:
subscription_status
no     4.2
yes    4.5
Name: satisfaction_rating, dtype: float64

Overall medians (the 'simple' option we are NOT using):
email_engagement:    51.4
satisfaction_rating: 4.4


### Step A — create flag columns *before* filling (otherwise we lose track of which rows were missing)

In [6]:
# 1 = the value was missing and has been filled in, 0 = original value
df["email_was_missing"] = df["email_engagement"].isnull().astype(int)
df["satisfaction_was_missing"] = df["satisfaction_rating"].isnull().astype(int)

print("Email flags:", df["email_was_missing"].sum())
print("Satisfaction flags:", df["satisfaction_was_missing"].sum())

Email flags: 6
Satisfaction flags: 8


### Step B — fill email engagement

In [7]:
# Medians for each group
email_median_sub    = df.loc[df["subscription_status"] == "yes", "email_engagement"].median().round(2)
email_median_nonsub = df.loc[df["subscription_status"] == "no",  "email_engagement"].median().round(2)

# Which rows need filling?
missing_sub    = (df["email_engagement"].isnull()) & (df["subscription_status"] == "yes")
missing_nonsub = (df["email_engagement"].isnull()) & (df["subscription_status"] == "no")

# Fill them
df.loc[missing_sub, "email_engagement"]    = email_median_sub
df.loc[missing_nonsub, "email_engagement"] = email_median_nonsub

print("Subscribers filled with", email_median_sub, "->", missing_sub.sum(), "rows")
print("Non-subscribers filled with", email_median_nonsub, "->", missing_nonsub.sum(), "rows")

cleaning_log.append(["email_engagement", "Missing values", missing_sub.sum() + missing_nonsub.sum(),
                     f"Filled with subscription-group median (yes={email_median_sub}, no={email_median_nonsub})",
                     "email_was_missing"])

Subscribers filled with 75.4 -> 3 rows
Non-subscribers filled with 38.35 -> 3 rows


### Step C — fill satisfaction rating (same method)

In [8]:
sat_median_sub    = df.loc[df["subscription_status"] == "yes", "satisfaction_rating"].median().round(2)
sat_median_nonsub = df.loc[df["subscription_status"] == "no",  "satisfaction_rating"].median().round(2)

missing_sub    = (df["satisfaction_rating"].isnull()) & (df["subscription_status"] == "yes")
missing_nonsub = (df["satisfaction_rating"].isnull()) & (df["subscription_status"] == "no")

df.loc[missing_sub, "satisfaction_rating"]    = sat_median_sub
df.loc[missing_nonsub, "satisfaction_rating"] = sat_median_nonsub

print("Subscribers filled with", sat_median_sub, "->", missing_sub.sum(), "rows")
print("Non-subscribers filled with", sat_median_nonsub, "->", missing_nonsub.sum(), "rows")

cleaning_log.append(["satisfaction_rating", "Missing values (not at random - mostly subscribers)",
                     missing_sub.sum() + missing_nonsub.sum(),
                     f"Filled with subscription-group median (yes={sat_median_sub}, no={sat_median_nonsub})",
                     "satisfaction_was_missing"])

Subscribers filled with 4.5 -> 6 rows
Non-subscribers filled with 4.2 -> 2 rows


In [9]:
# Check: no missing values should remain
print("Missing values left:", df.isnull().sum().sum())

Missing values left: 0


In [10]:
# Look at the filled rows to check they are sensible
filled_rows = (df["email_was_missing"] == 1) | (df["satisfaction_was_missing"] == 1)
df.loc[filled_rows, ["customer_id", "subscription_status", "email_engagement", "satisfaction_rating",
                     "email_was_missing", "satisfaction_was_missing"]]

,customer_id,subscription_status,email_engagement,satisfaction_rating,email_was_missing,satisfaction_was_missing
25,FG0026,no,38.35,2.6,1,0
62,FG0063,no,44.20,4.2,0,1
64,FG0065,yes,75.40,4.9,1,0
92,FG0093,yes,73.50,4.5,0,1
116,FG0117,yes,93.80,4.5,0,1
122,FG0123,yes,61.50,4.5,0,1
123,FG0124,no,38.35,4.9,1,0
154,FG0155,yes,75.40,4.7,1,0
156,FG0157,no,38.35,2.7,1,0
185,FG0186,yes,78.50,4.5,0,1


**Risk of this treatment:** every filled-in customer gets the *same* typical value, so we slightly understate how much customers vary,
and we assume a missing customer is "typical" of their group. Only 14 values are filled, so the effect is small — and the flag columns
let us test it in Step 3 (re-run the model without these rows and check the results don't change).

## 2.3 Outlier in distinct product types
One customer bought **12** different tea types; nobody else bought more than 9. It may be real, but a single extreme value can
distort a model with only 250 rows. We **cap** it at 9 (they stay a "very broad buyer") rather than delete the customer.

In [11]:
# Flag first, then cap
df["products_capped"] = (df["distinct_product_types"] > 9).astype(int)
print(df.loc[df["products_capped"] == 1, ["customer_id", "distinct_product_types"]])

df["distinct_product_types"] = df["distinct_product_types"].clip(upper=9)
print("\nNew maximum:", df["distinct_product_types"].max())

cleaning_log.append(["distinct_product_types", "Outlier (12 vs max 9 for others)", df["products_capped"].sum(),
                     "Capped at 9", "products_capped"])

    customer_id  distinct_product_types
204      FG0205                      12

New maximum: 9


## 2.4 Average order value of exactly £5.00
Two customers have an average order value of exactly £5.00 — the lowest value in the file, which looks like a
system floor or default. We cannot prove it is wrong, so we **keep and flag** these rows.

In [12]:
df["aov_suspicious"] = (df["average_order_value"] == 5.0).astype(int)
print(df.loc[df["aov_suspicious"] == 1, ["customer_id", "total_orders", "average_order_value", "customer_value_12m"]])

cleaning_log.append(["average_order_value", "Suspicious value (exactly £5.00)", df["aov_suspicious"].sum(),
                     "Kept, flagged", "aov_suspicious"])

    customer_id  total_orders  average_order_value  customer_value_12m
52       FG0053            10                  5.0               61.65
245      FG0246             1                  5.0                4.44


## 2.5 New customers (tenure < 12 months)
`customer_value_12m` is spend over the last 12 months. For a customer who joined 5 months ago, that is only **5 months** of spend,
so their value is understated compared with long-standing customers. We **flag** them so we can check our results without them.

In [13]:
df["new_customer"] = (df["tenure_months"] < 12).astype(int)
print("New customers (tenure < 12 months):", df["new_customer"].sum())

cleaning_log.append(["tenure_months", "Part-year value (tenure < 12 months)", df["new_customer"].sum(),
                     "Kept, flagged for sensitivity check", "new_customer"])

New customers (tenure < 12 months): 51


## 2.6 Logically inconsistent records
Some new customers have more than **2 orders per month of tenure** (e.g. 1 month as a customer, but 7 orders) — unlikely
for loose-leaf tea. Either tenure or orders is wrong, but we cannot tell which, so we **keep and flag**.

In [14]:
df["inconsistent_record"] = ((df["tenure_months"] < 12) &
                             (df["total_orders"] > 2 * df["tenure_months"])).astype(int)

print("Inconsistent records:", df["inconsistent_record"].sum())
df.loc[df["inconsistent_record"] == 1, ["customer_id", "tenure_months", "total_orders", "customer_value_12m"]]

Inconsistent records: 16


,customer_id,tenure_months,total_orders,customer_value_12m
0,FG0001,6,13,242.68
17,FG0018,1,7,156.29
18,FG0019,1,8,175.30
19,FG0020,4,9,125.79
29,FG0030,6,14,213.10
70,FG0071,8,27,1226.43
76,FG0077,1,3,213.83
100,FG0101,1,9,150.19
111,FG0112,1,3,200.52
137,FG0138,3,14,166.83


In [15]:
cleaning_log.append(["tenure_months / total_orders", "Logical inconsistency (> 2 orders per month of tenure)",
                     df["inconsistent_record"].sum(), "Kept, flagged for sensitivity check", "inconsistent_record"])

## 2.7 Create model-ready columns

**`is_subscriber`** — turns "yes"/"no" into 1/0 so models can use it.

**`high_value`** — our target. A customer is high-value if they are in the **top 20%** of 12-month spend.
Step 1 showed this group generates ~50% of revenue, so it is a meaningful business definition, not an arbitrary cut-off.

In [16]:
df["is_subscriber"] = (df["subscription_status"] == "yes").astype(int)

threshold = df["customer_value_12m"].quantile(0.80)
df["high_value"] = (df["customer_value_12m"] >= threshold).astype(int)

print("High-value threshold: £", round(threshold, 2))
print("High-value customers:", df["high_value"].sum(), "out of", len(df))

cleaning_log.append(["customer_value_12m", "Target variable created", df["high_value"].sum(),
                     f"high_value = 1 if value >= £{threshold:.2f} (top 20%)", "high_value"])

High-value threshold: £ 600.35
High-value customers: 50 out of 250


**Note:** `total_orders` and `average_order_value` stay in the file (useful for describing segments), but they will **not** be used
as drivers in Step 3, because value ≈ orders × average order value (circular).

## Final checks: before vs after

In [17]:
print("Missing values before:", raw.isnull().sum().sum())
print("Missing values after: ", df.isnull().sum().sum())
print("Rows before:", len(raw), "| Rows after:", len(df), "(no customers removed)")

Missing values before: 14
Missing values after:  0
Rows before: 250 | Rows after: 250 (no customers removed)


In [18]:
# Compare the columns we changed, before and after cleaning
changed = ["email_engagement", "satisfaction_rating", "distinct_product_types"]

compare = pd.DataFrame({
    "mean_before":   raw[changed].mean(),
    "mean_after":    df[changed].mean(),
    "median_before": raw[changed].median(),
    "median_after":  df[changed].median(),
})
compare.round(3)

,mean_before,mean_after,median_before,median_after
email_engagement,53.083,53.174,51.4,51.4
satisfaction_rating,4.157,4.165,4.4,4.4
distinct_product_types,4.500,4.488,4.0,4.0


In [19]:
# How many rows carry each flag?
flag_columns = ["email_was_missing", "satisfaction_was_missing", "products_capped",
                "aov_suspicious", "new_customer", "inconsistent_record"]
df[flag_columns].sum()

email_was_missing            6
satisfaction_was_missing     8
products_capped              1
aov_suspicious               2
new_customer                51
inconsistent_record         16
dtype: int64

## Save the clean data and the cleaning log

In [20]:
df.to_csv(CLEAN_FILE, index=False)

log = pd.DataFrame(cleaning_log, columns=["column", "issue", "rows_affected", "treatment", "flag_column"])
log.to_csv(LOG_FILE, index=False)

print("Saved:", CLEAN_FILE, "->", df.shape)
print("Saved:", LOG_FILE)
log

Saved: ../data_clean/ferngill_clean.csv -> (250, 21)
Saved: ../outputs/tables/02_cleaning_log.csv


,column,issue,rows_affected,treatment,flag_column
0,email_engagement,Missing values,6,Filled with subscription-group median (yes=75....,email_was_missing
1,satisfaction_rating,Missing values (not at random - mostly subscri...,8,Filled with subscription-group median (yes=4.5...,satisfaction_was_missing
2,distinct_product_types,Outlier (12 vs max 9 for others),1,Capped at 9,products_capped
3,average_order_value,Suspicious value (exactly £5.00),2,"Kept, flagged",aov_suspicious
4,tenure_months,Part-year value (tenure < 12 months),51,"Kept, flagged for sensitivity check",new_customer
5,tenure_months / total_orders,Logical inconsistency (> 2 orders per month of...,16,"Kept, flagged for sensitivity check",inconsistent_record
6,customer_value_12m,Target variable created,50,high_value = 1 if value >= £600.35 (top 20%),high_value


## Summary — what to say in the report
- **No customers were removed.** 14 missing values (6 email, 8 satisfaction) were filled using the median of the customer's
  subscription group, because missing satisfaction ratings were concentrated among subscribers.
- **Risk:** filled values reduce natural variation and assume a missing customer is "typical" of their group.
  Mitigation: flag columns allow a sensitivity check in Step 3.
- One product-range outlier was capped; suspicious £5.00 order values, part-year customers and inconsistent records were **flagged, not deleted**.
- Order count and order value are kept for description but excluded as drivers, because they make up the spend figure itself.